# CRISP-DM Phase 3: Data Preparation (Medical Dataset)

This notebook implements a leakage-safe, industry-style preparation pipeline for modeling.

Main objectives:
1. **Governed dataset copies:** keep one immutable source (`df_raw`) and one preparation copy (`df_prep`).
2. **Data quality enforcement:** validate schema and handle implausible vitals using explicit clinical rules.
3. **Feature engineering:** create clinically meaningful derived variables.
4. **Leakage prevention:** split first, then fit preprocessing (imputer/scaler) on train only.
5. **Reproducible export:** save processed matrices and metadata for downstream modeling.

In [8]:
import os
import json
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
import warnings

warnings.filterwarnings('ignore')
RANDOM_STATE = 42

## 1. Data Loading and Copy Governance

We keep dataset copies explicit and minimal:
- `df_raw`: immutable source snapshot
- `df_prep`: single working copy for Data Preparation

In [9]:
# Resolve base folder robustly whether notebook runs from project root or subfolder
candidate_roots = [Path.cwd(), Path.cwd() / 'classification_Medical_data _set']
base_dir = None
for root in candidate_roots:
    if (root / 'Medicaldataset.csv').exists():
        base_dir = root
        break

if base_dir is None:
    raise FileNotFoundError("Could not locate 'Medicaldataset.csv'.")

DATA_FILE = base_dir / 'Medicaldataset.csv'
df_raw = pd.read_csv(DATA_FILE)
df_prep = df_raw.copy(deep=True)

print(f"Base directory: {base_dir}")
print(f"Raw Data Loaded. Dimensions: {df_raw.shape}")
print(f"Preparation copy dimensions: {df_prep.shape}")

Base directory: C:\Users\rahma\Desktop\machine learning project\classification_Medical_data _set
Raw Data Loaded. Dimensions: (1319, 9)
Preparation copy dimensions: (1319, 9)


## 2. Schema Contract, Clinical Rules, and Feature Engineering

This block enforces:
- required-column validation
- explicit handling for implausible vitals (flag + set to missing)
- clinically meaningful derived features

In [10]:
required_columns = [
    'Age', 'Gender', 'Heart rate', 'Systolic blood pressure', 'Diastolic blood pressure',
    'Blood sugar', 'CK-MB', 'Troponin', 'Result'
]
missing_cols = [c for c in required_columns if c not in df_prep.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df_work = df_prep.copy(deep=True)

# Clinical plausibility rules (hard-invalid -> missing for later train-only imputation)
HR_MIN, HR_MAX = 25, 250
SBP_MIN, SBP_MAX = 50, 280

df_work['flag_hr_out_of_range'] = ((df_work['Heart rate'] < HR_MIN) | (df_work['Heart rate'] > HR_MAX)).astype(int)
df_work['flag_sbp_out_of_range'] = ((df_work['Systolic blood pressure'] < SBP_MIN) | (df_work['Systolic blood pressure'] > SBP_MAX)).astype(int)

df_work.loc[df_work['flag_hr_out_of_range'] == 1, 'Heart rate'] = np.nan
df_work.loc[df_work['flag_sbp_out_of_range'] == 1, 'Systolic blood pressure'] = np.nan

# Feature engineering
# 1) Pulse Pressure
# 2) CK_MB_Troponin_Ratio (safe denominator)
EPS = 1e-6
df_work['Pulse_Pressure'] = df_work['Systolic blood pressure'] - df_work['Diastolic blood pressure']
df_work['CK_MB_Troponin_Ratio'] = df_work['CK-MB'] / np.maximum(df_work['Troponin'], EPS)

print('Feature engineering and clinical rule enforcement complete.')
print('Flag counts:')
print(df_work[['flag_hr_out_of_range', 'flag_sbp_out_of_range']].sum())
display(df_work[['Pulse_Pressure', 'CK_MB_Troponin_Ratio']].head())

Feature engineering and clinical rule enforcement complete.
Flag counts:
flag_hr_out_of_range     5
flag_sbp_out_of_range    1
dtype: int64


,Pulse_Pressure,CK_MB_Troponin_Ratio
0,77.0,150.000000
1,52.0,6.367925
2,83.0,663.333333
3,65.0,113.688525
4,47.0,360.000000


## 3. Target Encoding and Train/Test Split

We use an explicit target mapping and stratified split.
All statistical preprocessing is fitted on training data only.

In [11]:
# Explicit target mapping (safer than implicit LabelEncoder ordering)
target_map = {'negative': 0, 'positive': 1}
df_work['Result'] = df_work['Result'].astype(str).str.strip().str.lower().map(target_map)
if df_work['Result'].isna().any():
    bad_labels = df_prep.loc[df_work['Result'].isna(), 'Result'].unique().tolist()
    raise ValueError(f"Unexpected target labels found: {bad_labels}")

X = df_work.drop(columns=['Result'])
y = df_work['Result'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print('Target mapping:', target_map)
print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")
print(f"Train positive rate: {y_train.mean():.4f}")
print(f"Test positive rate: {y_test.mean():.4f}")

Target mapping: {'negative': 0, 'positive': 1}
Training set shape: (1055, 12)
Testing set shape: (264, 12)
Train positive rate: 0.6142
Test positive rate: 0.6136


## 4. Train-Only Preprocessing (Imputation + Robust Scaling)

Continuous features are imputed and scaled using train-fitted statistics only.
Clinical rule flags are kept as binary features and not scaled.

In [12]:
flag_cols = ['flag_hr_out_of_range', 'flag_sbp_out_of_range']
continuous_cols = [c for c in X_train.columns if c not in flag_cols]

# Train-only fitting
imputer = SimpleImputer(strategy='median')
scaler = RobustScaler()

X_train_cont_imputed = pd.DataFrame(
    imputer.fit_transform(X_train[continuous_cols]),
    columns=continuous_cols,
    index=X_train.index
)
X_test_cont_imputed = pd.DataFrame(
    imputer.transform(X_test[continuous_cols]),
    columns=continuous_cols,
    index=X_test.index
)

X_train_cont_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_cont_imputed),
    columns=continuous_cols,
    index=X_train.index
)
X_test_cont_scaled = pd.DataFrame(
    scaler.transform(X_test_cont_imputed),
    columns=continuous_cols,
    index=X_test.index
)

# Add rule flags back without scaling
X_train_processed = pd.concat([X_train_cont_scaled, X_train[flag_cols].astype(int)], axis=1)
X_test_processed = pd.concat([X_test_cont_scaled, X_test[flag_cols].astype(int)], axis=1)

# Keep deterministic column order
ordered_cols = continuous_cols + flag_cols
X_train_processed = X_train_processed[ordered_cols]
X_test_processed = X_test_processed[ordered_cols]

print('Train-only preprocessing complete.')
print(f"Processed train shape: {X_train_processed.shape}")
print(f"Processed test shape: {X_test_processed.shape}")
display(X_train_processed.head())

Train-only preprocessing complete.
Processed train shape: (1055, 12)
Processed test shape: (264, 12)


,Age,Gender,Heart rate,Systolic blood pressure,Diastolic blood pressure,Blood sugar,CK-MB,Troponin,Pulse_Pressure,CK_MB_Troponin_Ratio,flag_hr_out_of_range,flag_sbp_out_of_range
881,-0.222222,0.0,-0.500000,-0.558824,-0.421053,0.083333,2.362694,-0.102273,-0.458333,3.738599,0,0
723,-1.222222,-1.0,-0.681818,0.382353,0.473684,-0.222222,0.344560,-0.125000,0.166667,2.020633,0,0
889,0.388889,0.0,0.136364,-0.058824,-0.736842,-0.125000,-0.323834,5.340909,0.500000,-0.339300,0,0
1004,0.777778,-1.0,0.909091,-0.058824,-0.263158,3.847222,-0.448187,0.022727,0.125000,-0.228354,0,0
761,-0.166667,-1.0,-0.272727,-0.235294,0.105263,0.388889,-0.383420,0.909091,-0.416667,-0.320471,0,0


## 5. Preparation QA Summary

This summary verifies data-preparation quality before export:
- no NaN/inf in processed matrices
- transparent counts of flagged records
- complete-case sensitivity sample size

In [13]:
def frame_has_inf(df_):
    return np.isinf(df_.to_numpy(dtype=float)).any()

qa_summary = {
    'train_rows': int(X_train_processed.shape[0]),
    'test_rows': int(X_test_processed.shape[0]),
    'feature_count': int(X_train_processed.shape[1]),
    'train_has_nan': bool(X_train_processed.isna().any().any()),
    'test_has_nan': bool(X_test_processed.isna().any().any()),
    'train_has_inf': bool(frame_has_inf(X_train_processed)),
    'test_has_inf': bool(frame_has_inf(X_test_processed)),
    'flag_hr_out_of_range_total': int(df_work['flag_hr_out_of_range'].sum()),
    'flag_sbp_out_of_range_total': int(df_work['flag_sbp_out_of_range'].sum())
}

# Optional sensitivity baseline: complete-case after hard-invalid conversion
complete_case_rows = int(df_work.drop(columns=['Result']).dropna().shape[0])
qa_summary['complete_case_rows_after_rule_based_nan'] = complete_case_rows
qa_summary['complete_case_pct_after_rule_based_nan'] = round(100 * complete_case_rows / len(df_work), 2)

print('QA Summary:')
display(pd.DataFrame([qa_summary]))

QA Summary:


,train_rows,test_rows,feature_count,train_has_nan,test_has_nan,train_has_inf,test_has_inf,flag_hr_out_of_range_total,flag_sbp_out_of_range_total,complete_case_rows_after_rule_based_nan,complete_case_pct_after_rule_based_nan
0,1055,264,12,False,False,False,False,5,1,1313,99.55


## 6. Export Processed Data and Metadata

We export processed datasets plus metadata to support reproducibility and clean handoff to modeling.

In [14]:
out_dir = base_dir / 'data' / 'processed' / 'medical'
out_dir.mkdir(parents=True, exist_ok=True)

# Main matrices
X_train_processed.to_csv(out_dir / 'X_train.csv', index=False)
X_test_processed.to_csv(out_dir / 'X_test.csv', index=False)
y_train.to_csv(out_dir / 'y_train.csv', index=False)
y_test.to_csv(out_dir / 'y_test.csv', index=False)

# Reproducibility metadata
metadata = {
    'random_state': RANDOM_STATE,
    'source_file': str(DATA_FILE),
    'input_shape': [int(df_raw.shape[0]), int(df_raw.shape[1])],
    'train_shape': [int(X_train_processed.shape[0]), int(X_train_processed.shape[1])],
    'test_shape': [int(X_test_processed.shape[0]), int(X_test_processed.shape[1])],
    'target_mapping': target_map,
    'continuous_columns': continuous_cols,
    'flag_columns': flag_cols,
    'clinical_bounds': {
        'Heart rate': [HR_MIN, HR_MAX],
        'Systolic blood pressure': [SBP_MIN, SBP_MAX]
    },
    'qa_summary': qa_summary
}

with open(out_dir / 'metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)

print(f"Success! Exported processed data to: {out_dir}")
print('Files:', [p.name for p in sorted(out_dir.glob('*'))])

Success! Exported processed data to: C:\Users\rahma\Desktop\machine learning project\classification_Medical_data _set\data\processed\medical
Files: ['metadata.json', 'X_test.csv', 'X_train.csv', 'y_test.csv', 'y_train.csv']
